In [ ]:
def extract_features_eda_ppg(df, participant_id):

    
    # Extract EDA features
    eda_features = extract_features_GSR(df, participant_id)

    if eda_features is None or len(eda_features) == 0:
        return None

   
    # Extract PPG features
    ppg_features = extract_features_ppg(df, participant_id)

    if ppg_features is None or len(ppg_features) == 0:
        return None

    
    # MERGE     
    merged = pd.merge(
        ppg_features,
        eda_features,
        on=["participant", "time_sec"],
        how="inner"
    )

    return merged

In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data/raw/Participants")
OUTPUT_DIR = Path("../data/processed/eda_ppg_testSOM")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------
# SELECT PARTICIPANTS
# -------------------------
participant_files = get_participant_files(DATA_DIR, 49, 50)

print("Participants loaded:", len(participant_files))

all_features = []
bad_participants = []

# -------------------------
# LOOP
# -------------------------
for file in participant_files:

    participant_id = file.parent.name
    print("\nProcessing:", participant_id)

    df = pd.read_csv(file)

    # -------------------------
    # BASIC GSR CHECK
    # -------------------------
    gsr = pd.to_numeric(df["gsr"], errors="coerce")

    if gsr.isna().mean() > 0.5:
        print("❌ Too many NaNs in GSR")
        bad_participants.append(participant_id)
        continue

    if gsr.std() < 1e-3:
        print("❌ Flat GSR signal")
        bad_participants.append(participant_id)
        continue

    # -------------------------
    # FEATURE EXTRACTION (EDA ONLY)
    # -------------------------
    features = extract_features_GSR(df, participant_id)

    if features is None or len(features) == 0:
        print("⚠️ No EDA features extracted")
        bad_participants.append(participant_id)
        continue

    # -------------------------
    # SAVE INDIVIDUAL
    # -------------------------
    output_file = OUTPUT_DIR / f"{participant_id}_eda_ppg_testSOM.csv"
    features.to_csv(output_file, index=False)

    print(f"✅ Saved: {output_file}")

    all_features.append(features)

# -------------------------
# COMBINE ALL
# -------------------------
if len(all_features) > 0:
    features_df = pd.concat(all_features, ignore_index=True)

    features_df.to_csv(OUTPUT_DIR / "eda_ppg_testSOM_all.csv", index=False)

    print("\nDONE ✅")
    print("Shape:", features_df.shape)

    # 🔍 QUICK STATS
    print("\nSC_RR summary:")
    print(features_df["SC_RR"].describe())

else:
    print("\n❌ No valid EDA features extracted")

# -------------------------
# SUMMARY
# -------------------------
print("\nBad participants skipped:", len(bad_participants))
print(bad_participants)